# 10.03 Runloop Sandbox

**`langchain-runloop`** 의 `RunloopSandbox` 는 Runloop 가 호스팅하는 **장수명 devbox**(컨테이너 + 영속 파일시스템)를 Deep Agents `SandboxBackendProtocol` 로 감싼 래퍼다.
Modal 이 "한 방에 떠서 일하고 사라지는" 서버리스라면, Runloop 은 **세션이 며칠 간 살아있는** 워크스페이스에 가깝다.

## 학습 목표

- `Runloop()` 클라이언트 → `client.devboxes.create_and_await_running(...)` → `RunloopSandbox(devbox=...)` 조립 패턴 이해
- 5대 파일 API(`read` · `write` · `upload_files` · `download_files` · `edit`) 와 `execute(cmd, timeout=...)` 사용
- `create_deep_agent(..., backend=RunloopSandbox(...))` 연동 — 에이전트의 셸·파일 도구가 전부 devbox 위로 라우팅
- **세션 스냅샷**(`devbox.snapshot.create`) 으로 환경을 동결·복원
- `idle_time_to_shutdown` / 명시 `shutdown` 라이프사이클

## 언제 쓰나 — Modal vs Daytona vs Runloop

| 선택 | 적합 | 특징 |
|------|------|------|
| Modal | 스테이트리스 burst, GPU | 서버리스, per-run 과금, 종료 시 휘발 |
| Daytona | Dev Container 기반 재현성, IDE 워크스페이스 | `.devcontainer.json` 호환, 스냅샷·볼륨 |
| **Runloop** | **장시간 세션**, 코드 리뷰·디버깅 루프, 에이전트가 같은 폴더에 며칠 머무는 경우 | devbox 가 **stop 후에도 파일시스템 보존**, 셸/브라우저 내장 |

## 10.03.1 환경 설정

필요 패키지: `langchain-runloop` (내부적으로 `runloop-api-client` + deep-agents 를 끌고 온다).
`.env` 에는 Runloop 콘솔에서 발급한 `RUNLOOP_API_KEY` 가 필요하다.

```bash
uv pip install langchain-runloop
# https://platform.runloop.ai 에서 API Key 발급 → .env 에 저장
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ.get("RUNLOOP_API_KEY"), "RUNLOOP_API_KEY 필요"

## 10.03.2 기본 사용 — devbox 한 대 띄우기

가장 짧은 E2E 경로.

1. `Runloop()` 클라이언트 생성 (`RUNLOOP_API_KEY` 환경변수 자동 사용).
2. `client.devboxes.create_and_await_running(...)` 로 devbox 한 대를 **running 상태까지 블로킹**.
3. `RunloopSandbox(devbox=raw)` 래퍼로 감싼 뒤 `.execute("python -c 'print(1+1)'")` 호출.
4. 명시 종료는 `client.devboxes.shutdown(devbox.id)` — 또는 `idle_time_to_shutdown` 으로 위임.

In [ ]:
from runloop_api_client import Runloop
from langchain_runloop import RunloopSandbox

client = Runloop()

raw = client.devboxes.create_and_await_running(
    name="lc-sandbox-hello",
    idle_time_to_shutdown=300,   # 5분 유휴 시 자동 stop
)

sandbox = RunloopSandbox(devbox=raw)
print("devbox id:", sandbox.id)

r = sandbox.execute("python3 -c 'print(1+1)'")
print("exit:", r.exit_code, "stdout:", r.output.strip())

client.devboxes.shutdown(raw.id)

## 10.03.3 `create_deep_agent(..., backend=RunloopSandbox(...))` 연동

Modal · Daytona 와 똑같이, Deep Agents 는 backend 를 **단일 객체**로 받아 `ls` · `read_file` · `write_file` · 셸 실행을 전부 Runloop devbox 로 라우팅한다.
Runloop devbox 는 stop 후에도 **파일시스템이 그대로 살아있어**, 같은 thread_id 의 멀티턴 코딩 세션에서 특히 유리하다.

In [ ]:
from deepagents import create_deep_agent

client = Runloop()
raw = client.devboxes.create_and_await_running(
    name="lc-sandbox-agent",
    idle_time_to_shutdown=600,
)
backend = RunloopSandbox(devbox=raw)

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[],
    system_prompt=(
        "너는 Runloop devbox 안에서 Python 스크립트를 작성·실행하는 엔지니어다. "
        "결과물은 /home/user/ 아래에 두고, 파일 목록과 주요 출력만 짧게 요약해서 답한다."
    ),
    backend=backend,
)

res = agent.invoke({
    "messages": [
        {"role": "user", "content": "fib.py 에 피보나치 10개를 출력하는 코드를 쓰고 실행해"}
    ]
})
print(res["messages"][-1].content)

client.devboxes.shutdown(raw.id)

## 10.03.4 파일 전송 · 라이프사이클

`RunloopSandbox` 의 파일 API 는 `ModalSandbox` · `DaytonaSandbox` 와 **동일 시그니처**다 (5종).

- `write(path, content: str)` · `read(path, offset=0, limit=2000)`
- `upload_files([(path, bytes), ...])` / `download_files([path, ...])`
- `edit(path, ...)` — 라인 단위 편집 (Deep Agents `edit_file` 도구 백엔드)
- 비동기 트윈도 모두 지원: `aexecute`, `awrite`, `aread`, `aupload_files`, `adownload_files`, `aedit`

라이프사이클 옵션:

| 필드 | 단위 | 의미 |
|------|------|------|
| `idle_time_to_shutdown` | 초 | 유휴 시 자동 shutdown — 컴퓨트 과금 중지 |
| `launch_parameters.resource_size_request` | 문자열 | `"SMALL"` · `"MEDIUM"` · `"LARGE"` 등 리소스 프리셋 |
| `blueprint_id` | str | 미리 빌드한 환경 스냅샷에서 즉시 시작 (cold start 단축) |

In [ ]:
client = Runloop()
raw = client.devboxes.create_and_await_running(
    name="lc-sandbox-fs",
    idle_time_to_shutdown=600,
)
sandbox = RunloopSandbox(devbox=raw)

# 1) 텍스트 한 개 쓰기
sandbox.write("/home/user/note.txt", "hello from host")

# 2) 배치 업로드 (bytes 만)
sandbox.upload_files([
    ("/home/user/data.csv", b"a,b\n1,2\n3,4\n"),
    ("/home/user/script.py",
     b"import csv; print(list(csv.reader(open('/home/user/data.csv'))))\n"),
])

# 3) 실행
r = sandbox.execute("python3 /home/user/script.py")
print("exec exit:", r.exit_code, "output:", r.output.strip())

# 4) 다운로드
downloaded = sandbox.download_files(["/home/user/note.txt"])
print("downloaded:", downloaded[0].content)

client.devboxes.shutdown(raw.id)

## 10.03.5 스냅샷 · 블루프린트

Runloop 의 강점은 **세션 스냅샷**이다. `pip install` · `apt-get install` 로 무거운 의존성을 깔아둔 devbox 의 디스크 상태를 한 번 찍어 두면, 다음에는 cold start 가 수 초 → 수백 ms 로 줄어든다.

두 가지 흐름:

### (a) Snapshot — 살아있는 devbox 의 상태를 즉석에서 동결

```python
snap = client.devboxes.snapshot_disk(raw.id, name='post-pip-install')
# 이후 새 devbox 시작 시 snapshot_id 로 복원
raw2 = client.devboxes.create_and_await_running(snapshot_id=snap.id)
```

### (b) Blueprint — 코드로 선언한 환경 빌드

```python
bp = client.blueprints.create_and_await_build_completed(
    name='ml-baseline',
    dockerfile="FROM python:3.12-slim\nRUN pip install pandas numpy",
)
raw3 = client.devboxes.create_and_await_running(blueprint_id=bp.id)
```

팀 공용 baseline (회사 내부 라이브러리 + 개발 도구) 을 blueprint 로 한 번 만들어 두고, 모든 에이전트 세션이 거기서 시작하는 패턴을 권장한다.

In [ ]:
# 스냅샷 시그니처만 시연 — 실제 디스크 스냅샷은 분 단위 시간을 잡아먹어
# 노트북 실행 시간을 늘리므로 호출은 주석 상태로 남긴다.
client = Runloop()
raw = client.devboxes.create_and_await_running(
    name="lc-sandbox-snap",
    idle_time_to_shutdown=300,
)
sandbox = RunloopSandbox(devbox=raw)

# 환경을 만들어 둔다
sandbox.execute("pip install --quiet requests")

# snap = client.devboxes.snapshot_disk(raw.id, name='with-requests')
# print('snapshot id:', snap.id)
# raw2 = client.devboxes.create_and_await_running(snapshot_id=snap.id)
# print('restored:', raw2.id)

print("스냅샷 호출은 비용/시간 절감을 위해 주석 처리 — 시그니처만 표기")
client.devboxes.shutdown(raw.id)

## 10.03.6 비용 · 지연 트레이드오프

| 축 | Runloop | Modal | Daytona |
|----|---------|-------|---------|
| 콜드스타트 | 수 초 (snapshot/blueprint 시 수백 ms) | 수 초 (이미지 캐시 시 < 1초) | 수 초 (스냅샷 시 수백 ms) |
| per-exec 지연 | ~100ms (devbox 내부 exec API) | ~100ms | ~100ms |
| 격리도 | devbox 컨테이너 | 컨테이너 (네트워크 차단 옵션) | 워크스페이스 컨테이너 |
| 파일시스템 | **stop 후에도 보존** | 종료 시 휘발 | stop 후 보존 → archive → delete |
| 비용 | 세션 시간 + 스토리지 | 실행 시간만 | 실행 + 스토리지 보관 |
| 적합 | **장시간 코드 리뷰·디버깅 루프** | 스테이트리스 burst, GPU | 재현성 있는 dev 환경 |

설계 힌트:

- 같은 thread_id 에 며칠씩 머무는 코딩 에이전트는 **Runloop**. devbox 가 살아있는 한 캐시된 가상환경·중간 산출물·Git checkout 이 그대로 남는다.
- `idle_time_to_shutdown` 은 공격적으로 잡고, 다음 사용자 메시지가 오면 다시 wake — 비용을 최소화한다.
- 신뢰 불가 코드는 devbox 자체가 격리되어 있지만, 추가로 `network_egress` 정책을 콘솔에서 정의하라.

## 체크리스트

- [ ] `.env` 에 `RUNLOOP_API_KEY` 존재
- [ ] `Runloop().devboxes.create_and_await_running(...)` 가 running 상태까지 블로킹 (`devboxes.create` 와 다름)
- [ ] `RunloopSandbox(devbox=raw)` — 키워드 인자 `devbox=` 만 받는다
- [ ] `idle_time_to_shutdown` 으로 유휴 자동 종료 — 명시 종료는 `client.devboxes.shutdown(id)`
- [ ] 무거운 의존성은 `snapshot_disk` 또는 `blueprint` 로 동결해 cold start 단축
- [ ] `create_deep_agent(..., backend=RunloopSandbox(devbox=raw))` — 에이전트 셸·파일이 devbox 로 라우팅

## 다음

- `04_deepagents/10_sandboxes_and_acp.ipynb` — Deep Agents × 샌드박스 실전
- `08_integration/10_sandboxes/01_modal.ipynb` — 서버리스 burst + GPU
- `08_integration/10_sandboxes/02_daytona.ipynb` — Dev Container 워크스페이스

## 참고

- `langchain-runloop` PyPI: https://pypi.org/project/langchain-runloop/
- Runloop API 문서: https://docs.runloop.ai
- Deep Agents backend protocol: `deepagents.backends.protocol.BackendProtocol`